# Reproduction notebook: 41_recover_downstream_significance_metadata_v3_clean_audit

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 200)

DATASETS = [
    "Solar",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "ETTh1",
]

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
    "SegMoE",
]

HORIZONS = [96, 192, 336, 720]

ROOT_CANDIDATES = [
    Path("/data/dataset/strong_forecaster"),
    Path("/data/strong_forecaster"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)

if ROOT is None:
    raise FileNotFoundError("strong_forecaster root not found")

OUT_DIR = ROOT / "significance_metadata_recovery_v3_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


In [ ]:
preferred = (
    ROOT
    / "four_backbone_dataset_meta_analysis"
    / "condition_level_selected.csv"
)

if not preferred.is_file():
    hits = sorted(ROOT.rglob("condition_level_selected.csv"))
    hits = [
        p for p in hits
        if "significance_metadata_recovery" not in str(p)
    ]

    if not hits:
        raise FileNotFoundError("condition_level_selected.csv not found")

    preferred = hits[0]

conditions = pd.read_csv(preferred)

required = {
    "Dataset",
    "Backbone",
    "Horizon",
    "MSEGain_pct",
}

missing = required - set(conditions.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

conditions = conditions[
    conditions["Dataset"].isin(DATASETS)
    & conditions["Backbone"].isin(BACKBONES)
    & conditions["Horizon"].astype(int).isin(HORIZONS)
].copy()

conditions["Horizon"] = conditions["Horizon"].astype(int)

KEY = ["Dataset", "Backbone", "Horizon"]

if len(conditions) != 96:
    raise RuntimeError(f"Expected 96 conditions; got {len(conditions)}")

if conditions.duplicated(KEY).any():
    raise RuntimeError("Duplicate selected conditions found")

print("Condition table:", preferred)
print("Conditions:", len(conditions))
print("Columns:", list(conditions.columns))


In [ ]:
SIG_COLS = [
    "CI_Low",
    "CI_High",
    "SignificantPositive",
    "SignificantNegative",
]

for c in SIG_COLS:
    if c not in conditions.columns:
        conditions[c] = np.nan

base = conditions.copy()

base["SelectedHasCI"] = (
    pd.to_numeric(base["CI_Low"], errors="coerce").notna()
    & pd.to_numeric(base["CI_High"], errors="coerce").notna()
)

print(
    "Significance already present in selected table:",
    int(base["SelectedHasCI"].sum()),
    "/96",
)

display(
    base.groupby("Dataset")
    .agg(
        Conditions=("Horizon", "size"),
        SelectedWithCI=("SelectedHasCI", "sum"),
    )
    .reset_index()
)


In [ ]:
def canon_dataset(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")
    return {
        "solar": "Solar",
        "solarenergy": "Solar",
        "weather": "Weather",
        "electricity": "Electricity",
        "ecl": "Electricity",
        "traffic": "Traffic",
        "exchange": "Exchange",
        "exchangerate": "Exchange",
        "etth1": "ETTh1",
    }.get(s, str(x).strip())


def canon_backbone(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")
    return {
        "patchtst": "PatchTST",
        "itransformer": "iTransformer",
        "timemixer": "TimeMixer",
        "segmoe": "SegMoE",
        "segmoeforecast": "SegMoE",
    }.get(s, str(x).strip())


def find_col(df, names):
    lower = {str(c).lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None


def as_bool(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return bool(v)
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return np.nan


def is_excluded_path(p):
    s = str(p)

    # Never read outputs from this recovery workflow back into itself.
    banned = [
        "significance_metadata_recovery/",
        "significance_metadata_recovery_v3_clean/",
    ]

    return any(token in s for token in banned)


def source_priority(p):
    name = p.name.lower()
    s = str(p).lower()

    if name == "bootstrap.csv":
        return 0

    if "with_bootstrap" in name or "bootstrap" in name:
        return 1

    # Derived evidence tables are useful only after true bootstrap files.
    if (
        "cross_backbone_integrated_evidence" in s
        or "three_backbone_integrated_evidence" in s
    ):
        return 3

    return 2


In [ ]:
candidate_files = []

for p in ROOT.rglob("*.csv"):
    if is_excluded_path(p):
        continue

    # The Experiment 37 selected table is handled separately above.
    if p.resolve() == preferred.resolve():
        continue

    try:
        size_mb = p.stat().st_size / (1024**2)
    except Exception:
        continue

    if size_mb > 500:
        continue

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(df, ["Dataset", "Data"])
    c_backbone = find_col(df, ["Backbone", "Model"])
    c_horizon = find_col(df, ["Horizon", "PredLen", "pred_len", "H"])
    c_lo = find_col(df, ["CI_Low", "CILow", "Lower", "Lower95", "CI95_Low"])
    c_hi = find_col(df, ["CI_High", "CIHigh", "Upper", "Upper95", "CI95_High"])

    if None in {c_dataset, c_backbone, c_horizon, c_lo, c_hi}:
        continue

    candidate_files.append({
        "Path": str(p),
        "Rows": len(df),
        "Priority": source_priority(p),
        "SizeMB": size_mb,
    })

inventory = pd.DataFrame(
    candidate_files,
    columns=["Path", "Rows", "Priority", "SizeMB"],
)

if len(inventory):
    inventory = (
        inventory
        .drop_duplicates("Path")
        .sort_values(["Priority", "Path"])
        .reset_index(drop=True)
    )

display(inventory)

inventory.to_csv(
    OUT_DIR / "clean_candidate_significance_files.csv",
    index=False,
)

print("Clean candidate files:", len(inventory))


In [ ]:
parsed_rows = []

for _, meta in inventory.iterrows():
    p = Path(meta["Path"])

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(df, ["Dataset", "Data"])
    c_backbone = find_col(df, ["Backbone", "Model"])
    c_horizon = find_col(df, ["Horizon", "PredLen", "pred_len", "H"])

    c_comp = find_col(df, ["Comparison", "Compare", "MethodComparison"])
    c_mean = find_col(df, [
        "MeanImprovement",
        "MeanDiff",
        "MSEImprovement",
        "DeltaMSE",
        "MeanDifference",
    ])

    c_lo = find_col(df, ["CI_Low", "CILow", "Lower", "Lower95", "CI95_Low"])
    c_hi = find_col(df, ["CI_High", "CIHigh", "Upper", "Upper95", "CI95_High"])

    c_pos = find_col(df, [
        "SignificantPositive",
        "SigPositive",
        "PositiveSignificant",
    ])

    c_neg = find_col(df, [
        "SignificantNegative",
        "SigNegative",
        "NegativeSignificant",
    ])

    c_block = find_col(df, ["BlockLen", "BlockLength", "BootstrapBlockLen"])
    c_rep = find_col(df, ["Replicates", "NBoot", "BootstrapReplicates"])

    for idx, r in df.iterrows():
        dataset = canon_dataset(r[c_dataset])
        backbone = canon_backbone(r[c_backbone])

        try:
            horizon = int(r[c_horizon])
        except Exception:
            continue

        if dataset not in DATASETS:
            continue
        if backbone not in BACKBONES:
            continue
        if horizon not in HORIZONS:
            continue

        lo = pd.to_numeric(
            pd.Series([r[c_lo]]),
            errors="coerce",
        ).iloc[0]

        hi = pd.to_numeric(
            pd.Series([r[c_hi]]),
            errors="coerce",
        ).iloc[0]

        if pd.isna(lo) or pd.isna(hi):
            continue

        mean = (
            pd.to_numeric(
                pd.Series([r[c_mean]]),
                errors="coerce",
            ).iloc[0]
            if c_mean is not None
            else np.nan
        )

        comparison = (
            str(r[c_comp]).strip()
            if c_comp is not None and not pd.isna(r[c_comp])
            else ""
        )

        parsed_rows.append({
            "Dataset": dataset,
            "Backbone": backbone,
            "Horizon": horizon,
            "Comparison": comparison,
            "MeanImprovement": mean,
            "CI_Low_recovered": lo,
            "CI_High_recovered": hi,
            "SigPositive_recovered": (
                as_bool(r[c_pos])
                if c_pos is not None
                else bool(lo > 0)
            ),
            "SigNegative_recovered": (
                as_bool(r[c_neg])
                if c_neg is not None
                else bool(hi < 0)
            ),
            "BlockLen_recovered": (
                r[c_block] if c_block is not None else np.nan
            ),
            "Replicates_recovered": (
                r[c_rep] if c_rep is not None else np.nan
            ),
            "RecoveredSourcePath": str(p),
            "RecoveredSourcePriority": int(meta["Priority"]),
            "RecoveredSourceRow": int(idx),
        })

parsed = pd.DataFrame(parsed_rows)

print("Parsed clean CI rows:", len(parsed))

parsed.to_csv(
    OUT_DIR / "all_clean_recovered_ci_rows.csv",
    index=False,
)


In [ ]:
def comparison_score(s):
    s = str(s).lower().strip()

    if not s:
        return 2

    direct = "direct" in s
    finalish = any(
        token in s
        for token in [
            "shrinkadaptive",
            "shrink",
            "adaptive",
            "final",
            "ours",
        ]
    )

    if direct and finalish:
        return 0
    if direct:
        return 1
    return 2


def sign_of(v, tol=1e-12):
    if pd.isna(v):
        return 0
    if v > tol:
        return 1
    if v < -tol:
        return -1
    return 0


gain_lookup = base.set_index(KEY)["MSEGain_pct"].to_dict()

resolved_rows = []

if len(parsed):
    parsed["ComparisonScore"] = parsed["Comparison"].map(comparison_score)

    for key, g in parsed.groupby(KEY):
        frozen_gain = gain_lookup.get(key, np.nan)
        frozen_sign = sign_of(frozen_gain)

        g = g.copy()

        g["IntervalSign"] = np.select(
            [
                g["CI_Low_recovered"] > 0,
                g["CI_High_recovered"] < 0,
            ],
            [
                1,
                -1,
            ],
            default=0,
        )

        # Nonsignificant intervals can coexist with either observed direction.
        g["SignConsistent"] = (
            (g["IntervalSign"] == 0)
            | (frozen_sign == 0)
            | (g["IntervalSign"] == frozen_sign)
        )

        g = g[g["SignConsistent"]].copy()

        if not len(g):
            continue

        g["RepNumeric"] = pd.to_numeric(
            g["Replicates_recovered"],
            errors="coerce",
        ).fillna(-1)

        g = g.sort_values(
            [
                "ComparisonScore",
                "RecoveredSourcePriority",
                "RepNumeric",
                "RecoveredSourcePath",
            ],
            ascending=[True, True, False, True],
        )

        resolved_rows.append(g.iloc[0].to_dict())

resolved = pd.DataFrame(resolved_rows)

print("Clean resolved historical conditions:", len(resolved))

if len(resolved):
    display(
        resolved.sort_values(KEY)
    )

resolved.to_csv(
    OUT_DIR / "resolved_clean_historical_significance.csv",
    index=False,
)


In [ ]:
if len(resolved):
    recovery_keep = [
        "Dataset",
        "Backbone",
        "Horizon",
        "CI_Low_recovered",
        "CI_High_recovered",
        "SigPositive_recovered",
        "SigNegative_recovered",
        "BlockLen_recovered",
        "Replicates_recovered",
        "RecoveredSourcePath",
        "RecoveredSourcePriority",
    ]

    rec = resolved[recovery_keep].copy()
else:
    rec = pd.DataFrame(
        columns=[
            "Dataset",
            "Backbone",
            "Horizon",
            "CI_Low_recovered",
            "CI_High_recovered",
            "SigPositive_recovered",
            "SigNegative_recovered",
            "BlockLen_recovered",
            "Replicates_recovered",
            "RecoveredSourcePath",
            "RecoveredSourcePriority",
        ]
    )

full = base.merge(
    rec,
    on=KEY,
    how="left",
    validate="one_to_one",
)

for c in ["CI_Low", "CI_High"]:
    full[c] = pd.to_numeric(full[c], errors="coerce")

full["Final_CI_Low"] = full["CI_Low"].combine_first(
    full["CI_Low_recovered"]
)

full["Final_CI_High"] = full["CI_High"].combine_first(
    full["CI_High_recovered"]
)

full["Final_SignificantPositive"] = (
    full["SignificantPositive"]
    .where(
        full["SignificantPositive"].notna(),
        full["SigPositive_recovered"],
    )
)

full["Final_SignificantNegative"] = (
    full["SignificantNegative"]
    .where(
        full["SignificantNegative"].notna(),
        full["SigNegative_recovered"],
    )
)

full["SignificanceSource"] = np.where(
    full["SelectedHasCI"],
    "Experiment37Selected",
    np.where(
        full["Final_CI_Low"].notna()
        & full["Final_CI_High"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["HasFinalCI"] = (
    full["Final_CI_Low"].notna()
    & full["Final_CI_High"].notna()
)

full["SignClass"] = np.select(
    [
        full["Final_CI_Low"] > 0,
        full["Final_CI_High"] < 0,
        full["HasFinalCI"],
    ],
    [
        "positive",
        "negative",
        "nonsignificant",
    ],
    default="missing",
)

full["ObservedDirection"] = np.select(
    [
        full["MSEGain_pct"] > 1e-12,
        full["MSEGain_pct"] < -1e-12,
    ],
    [
        "win",
        "loss",
    ],
    default="tie",
)

# Strict final sign audit.
bad = full[
    ((full["SignClass"] == "positive") & (full["MSEGain_pct"] <= 0))
    | ((full["SignClass"] == "negative") & (full["MSEGain_pct"] >= 0))
]

if len(bad):
    print("WARNING: sign-inconsistent final intervals:")
    display(bad[KEY + ["MSEGain_pct", "Final_CI_Low", "Final_CI_High", "SignificanceSource"]])
    raise RuntimeError("Final significance sign audit failed")

display(
    full.sort_values(KEY)
)

full.to_csv(
    OUT_DIR / "condition_level_96_with_clean_significance.csv",
    index=False,
)


In [ ]:
summary_ds = (
    full.groupby("Dataset")
    .agg(
        Conditions=("Horizon", "size"),
        WithCI=("HasFinalCI", "sum"),
        SignificantWins=("SignClass", lambda s: int((s == "positive").sum())),
        SignificantLosses=("SignClass", lambda s: int((s == "negative").sum())),
        NonSignificant=("SignClass", lambda s: int((s == "nonsignificant").sum())),
        Missing=("SignClass", lambda s: int((s == "missing").sum())),
    )
    .reset_index()
)

summary_bb = (
    full.groupby("Backbone")
    .agg(
        Conditions=("Horizon", "size"),
        WithCI=("HasFinalCI", "sum"),
        SignificantWins=("SignClass", lambda s: int((s == "positive").sum())),
        SignificantLosses=("SignClass", lambda s: int((s == "negative").sum())),
        NonSignificant=("SignClass", lambda s: int((s == "nonsignificant").sum())),
        Missing=("SignClass", lambda s: int((s == "missing").sum())),
    )
    .reset_index()
)

display(summary_ds)
display(summary_bb)

summary_ds.to_csv(
    OUT_DIR / "clean_significance_coverage_by_dataset.csv",
    index=False,
)

summary_bb.to_csv(
    OUT_DIR / "clean_significance_coverage_by_backbone.csv",
    index=False,
)

missing = full[~full["HasFinalCI"]].copy()

print("=" * 118)
print("EXPERIMENT 41 v3 — CLEAN SIGNIFICANCE AUDIT")
print("=" * 118)
print(f"Final CI coverage: {int(full['HasFinalCI'].sum())}/96")
print(f"Missing: {len(missing)}/96")
print(f"Significant wins: {int((full['SignClass'] == 'positive').sum())}")
print(f"Significant losses: {int((full['SignClass'] == 'negative').sum())}")
print(f"Non-significant: {int((full['SignClass'] == 'nonsignificant').sum())}")

print("\nSource breakdown:")
print(full["SignificanceSource"].value_counts(dropna=False).to_string())

print("\nDataset summary:")
print(summary_ds.to_string(index=False))

if len(missing):
    print("\nStill missing:")
    print(
        missing[
            KEY + ["MSEGain_pct", "ObservedDirection"]
        ].sort_values(KEY).to_string(index=False)
    )


In [ ]:
paper = full[
    [
        "Dataset",
        "Backbone",
        "Horizon",
        "MSEGain_pct",
        "Final_CI_Low",
        "Final_CI_High",
        "SignClass",
        "SignificanceSource",
    ]
].copy()

paper = paper.sort_values(
    ["Dataset", "Backbone", "Horizon"]
)

display(paper)

paper.to_csv(
    OUT_DIR / "paper_downstream_significance_96_conditions.csv",
    index=False,
)

latex = paper.copy()

for c in [
    "MSEGain_pct",
    "Final_CI_Low",
    "Final_CI_High",
]:
    latex[c] = latex[c].map(
        lambda v: "" if pd.isna(v) else f"{v:.6f}"
    )

(OUT_DIR / "paper_downstream_significance_96_conditions.tex").write_text(
    latex.to_latex(
        index=False,
        escape=False,
    ),
    encoding="utf-8",
)

print("Saved final CSV and LaTeX table.")
